# ML Pipeline - XGBoost pour projection sur nouveaux hôtels

**Objectif** : Entraîner **uniquement sur les 5 hôtels pivots**, séparer strictement variables cibles / descriptives, puis générer le profil complet des ~286 variables cibles (CA + nombre de ventes mensuelles croisées TYPE × GAMME) pour **n'importe quel nouvel hôtel**.

Un nouvel hôtel peut avoir :
- n'importe quel nombre de chambres
- n'importe quelle adresse (→ POI et météo différents)
- n'importe quelles autres valeurs dans les variables descriptives ROD

**Rien de spécial sur `nb_de_chambres`** — c'était juste un exemple dans la discussion.

Pour un nouvel hôtel :
1. Construire **son** vecteur complet de features descriptives (même colonnes que pendant l'entraînement).
   - Idéalement via enrichissement réel (geocoding + POI + météo avec `enrich_hotel.py`) + infos ROD connues (nb chambres, marque, etc.).
   - En dernier recours / pour démo rapide : moyenne des 5 pivots + overrides sur les valeurs connues.
2. `pred = model.predict( scaler.transform( new_X_row ) )` → vecteur 286 complet.

Le modèle apprend à mapper **n'importe quel profil descriptif** → profil de ventes (saisonnalité, mix F&B, distribution par catégorie).

- Variables cibles = toutes les colonnes `__montant` ou `__nbr_ventes`
- Features descriptives uniquement → **zéro fuite**
- Modèle : `MultiOutputRegressor(XGBRegressor)` (forte régularisation, adapté à N=5)


In [10]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputRegressor
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')

df = pd.read_excel('ml_data.xlsx')
print("Hôtels pivots :", df["HOTEL_NAME"].tolist())

target_cols = [c for c in df.columns if '__montant' in c or '__nbr_ventes' in c]
print('Variables cibles identifiées :', len(target_cols), "(montants + nbr_ventes croisés)")

# === SÉPARATION STRICTE (le point le plus important) ===
# X = TOUTES les variables descriptives (POI, météo, ROD aplati, nb_chambres, marque, etc.)
# y = UNIQUEMENT les 286 cibles de ventes/CA
X = df.drop(columns=target_cols + ['HOTEL_NAME'], errors='ignore').select_dtypes(include=[np.number]).fillna(0)
y = df[target_cols].fillna(0)

print('X (features descriptives):', X.shape)
print('y (cibles):', y.shape)

overlap = set(X.columns) & set(target_cols)
assert len(overlap) == 0, f'FUITE ! {overlap}'
print('✅ Séparation validée : aucune variable cible dans les features.')
print('Toute colonne de X peut être fournie / modifiée pour un nouvel hôtel.')


Hôtels pivots : ['Ibis budget Nice', 'Ibis budget Strasbourg Centre République', 'Mercure Paris Montmartre Sacré-Cœur', 'Novotel Megève Mont-Blanc', 'Novotel Paris Tour Eiffel']
Variables cibles identifiées : 286 (montants + nbr_ventes croisés)
X (features descriptives): (5, 5169)
y (cibles): (5, 286)
✅ Séparation validée : aucune variable cible dans les features.
Toute colonne de X peut être fournie / modifiée pour un nouvel hôtel.


In [11]:
# === Réduction de dimension (N=5 impose une réduction forte) ===
# On sélectionne les features les plus corrélées avec le CA total.
# AUCUNE colonne n'est forcée ou boostée : n'importe quelle variable descriptive
# peut différer pour un nouvel hôtel (chambres, POI, météo, autres ROD...).

y_tot = y[[c for c in target_cols if '__montant' in c]].sum(axis=1)
corrs = X.corrwith(y_tot).abs().sort_values(ascending=False)

n_features = 350
top = list(corrs.head(n_features).index)
X_red = X[top].copy()

scaler = StandardScaler()
X_s = scaler.fit_transform(X_red)

print('Features après réduction :', X_red.shape[1])
print('Top 5 features les plus corrélées avec CA total :')
print(list(corrs.head(5).index))
print()
print('Note : la réduction est purement basée sur la corrélation globale.')
print('       Si une feature est importante pour un nouvel hôtel, elle sera utilisée si présente.')


Features après réduction : 350
Top 5 features les plus corrélées avec CA total :
['etape_rod__2_services_equipements__sous_etape_rod__f_b__data__restaurant', 'etape_rod__2_services_equipements__sous_etape_rod__non_f_b__data__salles_de_reunion', 'm04_rhum_median.2', 'm04_rhum_median.8', 'm04_rhum_median.5']

Note : la réduction est purement basée sur la corrélation globale.
       Si une feature est importante pour un nouvel hôtel, elle sera utilisée si présente.


In [13]:
# === Modèle XGBoost multi-output (entraîné uniquement sur les 5 pivots) ===
xgb_params = dict(
    n_estimators=80,
    max_depth=3,
    learning_rate=0.04,
    reg_alpha=0.8,
    reg_lambda=1.0,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=1,
    random_state=42,
    verbosity=0
)

model = MultiOutputRegressor(XGBRegressor(**xgb_params))
model.fit(X_s, y)

print('✅ Modèle entraîné.')
print("   5 exemples d'entraînement (pivots) → 286 cibles.")
print("   Le modèle peut maintenant recevoir le vecteur descriptif de N'IMPORTE QUEL hôtel.")


✅ Modèle entraîné.
   5 exemples d'entraînement (pivots) → 286 cibles.
   Le modèle peut maintenant recevoir le vecteur descriptif de N'IMPORTE QUEL hôtel.


In [14]:
# === Génération du profil de ventes (286 cibles) pour un nouvel hôtel ===
# RÈGLE IMPORTANTE : 
# On n'utilise **jamais** la moyenne (ou toute statistique) des 5 pivots 
# pour inventer les features descriptives d'un nouvel hôtel.
# Cela n'a aucun sens : un nouvel hôtel a ses propres caractéristiques (adresse, POI, météo, etc.).
#
# Pour un nouvel hôtel tu dois construire le vecteur new_X avec ses VRAIES valeurs.
# Le code ci-dessous est uniquement une démo "faible" (on part d'un pivot proche).

nb_ch_col = [c for c in X_red.columns if 'nb_de_chambres' in c][0]

# Pour la démo on choisit comme base le pivot dont le nb_ch est le plus proche de 180.
# (Ibis budget Nice = 129). 
# En production tu remplaces complètement ce vecteur par les données du nouvel hôtel.
dists = (X[nb_ch_col] - 180).abs()
closest_idx = dists.idxmin()
base_row = X_red.loc[[closest_idx]].copy()

new_X = base_row.copy()
new_X[nb_ch_col] = 180

new_s = scaler.transform(new_X)
p = model.predict(new_s)[0]
pred = pd.DataFrame([p], columns=target_cols)

mont_cols = [c for c in target_cols if '__montant' in c]
ca_tot = round(pred[mont_cols].sum(axis=1).values[0])

print("ml.ipynb mis à jour")
print("Cibles séparées : 286 colonnes (__montant + __nbr_ventes croisées)")
print("X = features descriptives uniquement")
print("Modèle : MultiOutputRegressor(XGBRegressor) sur les 5 pivots")
print("")
print("Exemple (très limité) : 180 chambres + template du pivot le plus proche (pas de moyenne !)")
print("CA annuel total généré :", ca_tot)
print("(vecteur complet des 286 cibles)")
print("")
print("⚠️  ATTENTION : ceci reste une démo.")
print("    En réalité tu fournis le vecteur descriptif complet du nouvel hôtel")
print("    (via enrich_hotel.py pour l\'adresse + infos ROD).")
print("    Il n\'y a aucune dépendance aux statistiques globales des 5 pivots.")


ml.ipynb mis à jour
Cibles séparées : 286 colonnes (__montant + __nbr_ventes croisées)
X = features descriptives uniquement
Modèle : MultiOutputRegressor(XGBRegressor) sur les 5 pivots

Exemple (très limité) : 180 chambres + template du pivot le plus proche (pas de moyenne !)
CA annuel total généré : 23710
(vecteur complet des 286 cibles)

⚠️  ATTENTION : ceci reste une démo.
    En réalité tu fournis le vecteur descriptif complet du nouvel hôtel
    (via enrich_hotel.py pour l'adresse + infos ROD).
    Il n'y a aucune dépendance aux statistiques globales des 5 pivots.


In [15]:
# === Comment faire pour un vrai nouvel hôtel (n'importe quelles caractéristiques) ===
# Le plus important : construire un vecteur de features qui corresponde à CE nouvel hôtel.
#
# 1. Récupérer les features réelles quand c'est possible :
#    - Adresse / ville → geocode → lat/lon
#    - POI (commerces à proximité) + météo → via enrich_hotel.py
#    - Infos ROD connues : nombre de chambres, marque, etc.
#
# 2. Mettre ces valeurs dans un DataFrame avec exactement les colonnes de X_red.
#
# 3. Prédire.

print("=== Méthode générale pour n'importe quel nouvel hôtel ===")
print()
print("new_X = pd.DataFrame([ {")
print("    'etape_rod__...__nb_de_chambres': 180,     # ou 140, 250, etc.")
print("    'fb_0_3km': 45, 'not_fb_0_3km': 120,       # POI réels pour son adresse")
print("    'm01_dwpt_mean': 4.2, ...                  # stats météo réelles")
print("    # ... toutes les autres features descriptives")
print("}], columns=X_red.columns)")
print()
print("pred = model.predict( scaler.transform(new_X) )")
print()
print("→ pred contient les 286 valeurs de CA et ventes par mois/type/gamme")
print()
print("Le 'mean + override nb_ch' n'est qu'un raccourci quand on ne sait presque rien.")
print("Dès qu'on a une adresse, il faut utiliser l'enrichissement pour les POI + météo.")


=== Méthode générale pour n'importe quel nouvel hôtel ===

new_X = pd.DataFrame([ {
    'etape_rod__...__nb_de_chambres': 180,     # ou 140, 250, etc.
    'fb_0_3km': 45, 'not_fb_0_3km': 120,       # POI réels pour son adresse
    'm01_dwpt_mean': 4.2, ...                  # stats météo réelles
    # ... toutes les autres features descriptives
}], columns=X_red.columns)

pred = model.predict( scaler.transform(new_X) )

→ pred contient les 286 valeurs de CA et ventes par mois/type/gamme

Le 'mean + override nb_ch' n'est qu'un raccourci quand on ne sait presque rien.
Dès qu'on a une adresse, il faut utiliser l'enrichissement pour les POI + météo.


In [16]:
# === Rappel important ===
# Aucune dépendance aux statistiques des 5 pivots pour décrire un nouvel hôtel.
#
# new_X pour un nouvel hôtel doit venir de :
#   - enrich_hotel (adresse → lat/lon → POI + weather)
#   - tes données ROD (nb chambres, marque, équipements, etc.)
#   - toute autre source qui décrit CE nouvel hôtel
#
# Exemple de construction propre (sans aucune moyenne des pivots) :
# new_X = pd.DataFrame([{
#     nb_ch_col: 180,
#     "fb_0_1km": 12,
#     "fb_0_5km": 85,
#     # ... toutes les colonnes de X_red avec les vraies valeurs
# }], columns=X_red.columns)
#
# pred = model.predict(scaler.transform(new_X))


## Compréhension Métier : Conversion & Réallocation

Le modèle prédit le profil. La couche métier gère la cohérence et les scénarios business (ex: plus d'espace si on supprime l'alcool).

In [ ]:
import business_logic as biz
print('Exemple métier avec 100 clients (logique ROD + ML)')
# Assume a prediction gave certain by_gamme
example_by_gamme = {'ALCOOL': 400, 'FOOD_SALEE': 3000, 'FOOD_SUCREE': 2500, 'SANS_ALCOOL': 1500, 'ACCESSOIRES': 3000}
total = sum(example_by_gamme.values())
print('Base CA total:', total)
print('Mix % :', biz.profile_to_mix({k: v for k,v in example_by_gamme.items()}))  # wait, adapt

# Reallocation example
new_mix = biz.reallocate_mix(example_by_gamme, {'ALCOOL': 0})
print('Après suppression alcool (réallocation prop.):', new_mix)
print('Nouveau total:', sum(new_mix.values()))

# Funnel
f = biz.compute_funnel(180, 0.78)
print('Funnel pour 180ch @78% TO:', f)

# P&L example
pnl = biz.simulate_pnl(180, 5.0, 'SIMPLY', example_by_gamme)
print('P&L simple:', pnl)

### Note sur les inputs

Pour une version future, on peut passer les features en 'mix % souhaité' plutôt qu'absolu pour mieux prédire les formes de vente. Pour l'instant, la prédiction ML donne la forme naturelle, et le business_logic gère les ajustements voulus par le directeur.

## Modification du modèle pour les % de mix (selon le métier)

Le directeur ne saisit pas POI/météo (auto). Il saisit infos hôtel + son choix de % F&B / sous-catégories.

Le modèle doit supporter :
- Proposer le mix naturel (meilleur selon les pivots)
- Prédire avec mix forcé par le directeur, de façon cohérente (somme = 100%)

On ne change pas le modèle ML de base (il prédit le profil naturel), mais on modifie la **partie inférence** pour intégrer le mix % en entrée et garantir la cohérence en sortie.

In [ ]:
import business_logic as biz
import json

# Simulation de ce que fait le serveur maintenant
print('=== Modification ML pour supporter les % mix du directeur ===\n')

# 1. On simule les inputs du directeur (ROD + %)
nb_ch = 180
to = 0.78
desired_mix = {'ALCOOL': 0.0, 'FOOD_SALEE': 0.4, 'FOOD_SUCREE': 0.35, 'SANS_ALCOOL': 0.25}  # exemple

# 2. Volume (pas dans le ML, calculé par formules ROD)
volume = biz.compute_volume_from_rod(nb_ch, to)
print(f'Volume estimé (acheteurs/mois) : {volume}')

# 3. Le modèle ML donne le profil 'naturel' (sur les features de l'hôtel)
#    (dans le vrai code : build vec from enriched + ROD inputs, predict)
print('\nProfil naturel (exemple de ce que donnerait le ML sur les features) :')
natural_profile = {'ALCOOL': 450, 'FOOD_SALEE': 3200, 'FOOD_SUCREE': 2800, 'SANS_ALCOOL': 2100, 'ACCESSOIRES': 2800}
nat_total = sum(natural_profile.values())
print('Total naturel ML :', round(nat_total))
nat_mix = {k: round(v/nat_total*100,1) for k,v in natural_profile.items()}
print('Mix naturel % :', nat_mix)

# 4. Si le directeur force un mix (en %)
print('\n--- Directeur force un mix (ex: plus de FOOD_SALEE, 0 alcool) ---')
coherent = biz.predict_coherent_with_mix(natural_profile, desired_mix, nat_total)
print('Profil ajusté (cohérent) :', {k:round(v) for k,v in coherent.items()})
print('Nouveau total :', round(sum(coherent.values())))

# 5. Gain / projection
print('\nLe modèle (via réallocation) permet de voir le gain en CA en forçant ce mix.')
print('Dans l\'app : on utilise aussi le volume + m_lin pour le P&L complet.')

# Idée future : ajouter les % désirés comme features d'entrée du modèle
# (pour que le ML apprenne directement les impacts de différents mixes)
# Pour l'instant on le fait en post-traitement pour garder la cohérence.